⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# PRXBI_DW.WC_BADGE_DETAILS_D Data Load\n",
        "\n",
        "**Source File:** `WC_BADGE_DETAILS_D.txt`\n",
        "**Conversion Date:** 2023-10-27"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "from delta.tables import DeltaTable\n",
        "from pyspark.sql import functions as F\n",
        "from pyspark.sql.types import (\n",
        "    StructType, StructField,\n",
        "    StringType, LongType, IntegerType, DoubleType,\n",
        "    DecimalType, TimestampType, DateType, BinaryType, FloatType\n",
        ")\n",
        "from pyspark.sql.window import Window\n",
        "from pyspark.sql import SparkSession\n",
        "\n",
        "spark = SparkSession.builder.getOrCreate()"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"380\") # Default value for DATASOURCE_NUM_ID if not provided\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"-1\")       # Default value for ETL_PROC_WID if not provided\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"-1\")        # Default value for ODI_SESS_NO if not provided\n",
        "dbutils.widgets.text(\"v_ETL_JOB_TYPE\", \"ETL_AMERCURY_BADGE\") # Default value for v_ETL_JOB_TYPE\n",
        "\n",
        "datasource_num_id = int(dbutils.widgets.get(\"DATASOURCE_NUM_ID\"))\n",
        "etl_proc_wid      = int(dbutils.widgets.get(\"ETL_PROC_WID\"))\n",
        "odi_sess_no       = dbutils.widgets.get(\"ODI_SESS_NO\")\n",
        "v_etl_job_type    = dbutils.widgets.get(\"v_ETL_JOB_TYPE\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## ETL Parameters"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {2}, {3}, {4}, {5}, {6}\n",
        "# Extract ETL parameters from wc_etl_parameters table\n",
        "etl_parameters_df = (\n",
        "    spark.table(\"workspace.prxbi_dw.wc_etl_parameters\")\n",
        "    .filter(F.col(\"ETL_JOB_TYPE\") == F.lit(v_etl_job_type))\n",
        ")\n",
        "\n",
        "etl_last_extract_time = (\n",
        "    etl_parameters_df.select(F.col(\"etl_last_extract_time\")).collect()[0][0]\n",
        ")\n",
        "\n",
        "etl_current_extract_time = (\n",
        "    etl_parameters_df.select(F.col(\"etl_current_extract_time\")).collect()[0][0]\n",
        ")\n",
        "\n",
        "wc_etl_row_wid = (\n",
        "    etl_parameters_df.select(F.col(\"ROW_WID\")).collect()[0][0]\n",
        ")\n",
        "\n",
        "print(f\"ETL Last Extract Time: {etl_last_extract_time}\")\n",
        "print(f\"ETL Current Extract Time: {etl_current_extract_time}\")\n",
        "print(f\"WC ETL Row WID: {wc_etl_row_wid}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Staging Table: c_mercury_badge_ts_stg"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {30}\n",
        "# Drop the staging table if it exists\n",
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.c_mercury_badge_ts_stg PURGE\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {40}, {50}\n",
        "# Build and write the staging DataFrame\n",
        "\n",
        "# Subquery for deduplication: AMERCURY_BADGE_TS_2\n",
        "am_badge_ts_dedup_df = (\n",
        "    spark.table(\"workspace.prxbi_ts.wc_mercury_badge_ts\")\n",
        "    .filter(\n",
        "        (F.col(\"INT_INSERT_DATE\") > F.lit(etl_last_extract_time))\n",
        "        & (F.col(\"INT_INSERT_DATE\") <= F.lit(etl_current_extract_time))\n",
        "    )\n",
        "    .withColumn(\n",
        "        \"rn\",\n",
        "        Window.partitionBy(\"ID\").orderBy(\n",
        "            F.col(\"INT_INSERT_DATE\").desc(), F.col(\"VERSIONNUMBER\").desc()\n",
        "        ).row_number()\n",
        "    )\n",
        "    .filter(F.col(\"rn\") == 1)\n",
        "    .select(\n",
        "        F.col(\"ID\").alias(\"ID_DEDUP\"),\n",
        "        F.col(\"INT_INSERT_DATE\").alias(\"INT_INSERT_DATE_DEDUP\"),\n",
        "        F.col(\"VERSIONNUMBER\").alias(\"VERSIONNUMBER_DEDUP\")\n",
        "    )\n",
        ")\n",
        "\n",
        "# Main staging query\n",
        "staging_df = (\n",
        "    spark.table(\"workspace.prxbi_ts.wc_mercury_badge_ts\").alias(\"AMERCURY_BADGE_TS\")\n",
        "    .join(\n",
        "        am_badge_ts_dedup_df.alias(\"AMERCURY_BADGE_TS_2\"),\n",
        "        (\n",
        "            (F.col(\"AMERCURY_BADGE_TS.INT_INSERT_DATE\") == F.col(\"AMERCURY_BADGE_TS_2.INT_INSERT_DATE_DEDUP\"))\n",
        "            & (F.col(\"AMERCURY_BADGE_TS.VERSIONNUMBER\") == F.col(\"AMERCURY_BADGE_TS_2.VERSIONNUMBER_DEDUP\"))\n",
        "            & (F.col(\"AMERCURY_BADGE_TS.ID\") == F.col(\"AMERCURY_BADGE_TS_2.ID_DEDUP\"))\n",
        "        ),\n",
        "        \"inner\"\n",
        "    )\n",
        "    .select(\n",
        "        F.col(\"AMERCURY_BADGE_TS.ID\").cast(StringType()).alias(\"ID\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.BADGELOCATION\").cast(StringType()).alias(\"BADGELOCATION\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.BADGETOKEN\").cast(StringType()).alias(\"BADGETOKEN\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.BADGEVERSION\").cast(LongType()).alias(\"BADGEVERSION\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.CONTACTEMAIL\").cast(StringType()).alias(\"CONTACTEMAIL\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.CONTACTFIRSTNAME\").cast(StringType()).alias(\"CONTACTFIRSTNAME\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.CONTACTJOBTITLE\").cast(StringType()).alias(\"CONTACTJOBTITLE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.CONTACTLASTNAME\").cast(StringType()).alias(\"CONTACTLASTNAME\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.CONTACTPERSONRXMASTERID\").cast(StringType()).alias(\"CONTACTPERSONRXMASTERID\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.CREATEDBYREGISTRATIONTYPE\").cast(StringType()).alias(\"CREATEDBYREGISTRATIONTYPE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.CREATEDBYTYPE\").cast(StringType()).alias(\"CREATEDBYTYPE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.CULTURE\").cast(StringType()).alias(\"CULTURE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.CUSTOMERTYPE\").cast(StringType()).alias(\"CUSTOMERTYPE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.EVENTEDITIONGBSCODE\").cast(StringType()).alias(\"EVENTEDITIONGBSCODE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.ISBADGEUPDATE\").cast(StringType()).alias(\"ISBADGEUPDATE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.MARKETINGPREFERENCESPROMPTREQUIRED\").cast(StringType()).alias(\"MARKETINGPREFERENCESPROMPTREQU\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.ORGANISATIONCITY\").cast(StringType()).alias(\"ORGANISATIONCITY\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.ORGANISATIONCOUNTRYCODE\").cast(StringType()).alias(\"ORGANISATIONCOUNTRYCODE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.ORGANISATIONDISPLAYNAME\").cast(StringType()).alias(\"ORGANISATIONDISPLAYNAME\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.ORGANISATIONRXMASTERID\").cast(StringType()).alias(\"ORGANISATIONRXMASTERID\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.ORGANISATIONSTATE\").cast(StringType()).alias(\"ORGANISATIONSTATE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.PARTICIPATINGORGANISATIONID\").cast(StringType()).alias(\"PARTICIPATINGORGANISATIONID\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.PRODUCTCODE\").cast(StringType()).alias(\"PRODUCTCODE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.QRCODECONTENT\").cast(StringType()).alias(\"QRCODECONTENT\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.REGISTRATIONID\").cast(StringType()).alias(\"REGISTRATIONID\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.STATUS\").cast(LongType()).alias(\"STATUS\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.SUPPORTSTAFFCOMPANYADDRESS\").cast(StringType()).alias(\"SUPPORTSTAFFCOMPANYADDRESS\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.SUPPORTSTAFFCOMPANYNAME\").cast(StringType()).alias(\"SUPPORTSTAFFCOMPANYNAME\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.SUPPORTSTAFFMOBILEPHONE\").cast(StringType()).alias(\"SUPPORTSTAFFMOBILEPHONE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.SUPPORTSTAFFREPORTSTO\").cast(StringType()).alias(\"SUPPORTSTAFFREPORTSTO\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.SUPPORTSTAFFSTANDS\").cast(StringType()).alias(\"SUPPORTSTAFFSTANDS\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.SUPPORTSTAFFUSERACCESS\").cast(StringType()).alias(\"SUPPORTSTAFFUSERACCESS\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.VERSIONNUMBER\").cast(LongType()).alias(\"VERSIONNUMBER\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.MOBILEPHONE\").cast(StringType()).alias(\"MOBILEPHONE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.FIRSTSCANNEDDATE\").cast(TimestampType()).alias(\"FIRSTSCANNEDDATE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.LASTPRINTEDDATE\").cast(TimestampType()).alias(\"LASTPRINTEDDATE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.ACCESSVALIDITYMODIFIEDDATE\").cast(TimestampType()).alias(\"ACCESSVALIDITYMODIFIEDDATE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.CREATEDDATE\").cast(TimestampType()).alias(\"CREATEDDATE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.COMPANYPRODUCTCODE\").cast(StringType()).alias(\"COMPANYPRODUCTCODE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.PAYMENTSTATUS\").cast(StringType()).alias(\"PAYMENTSTATUS\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.PHOTOKEY\").cast(StringType()).alias(\"PHOTOKEY\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.PHOTOSOURCE\").cast(StringType()).alias(\"PHOTOSOURCE\"),\n",
        "        F.col(\"AMERCURY_BADGE_TS.PHOTOSOURCETYPE\").cast(StringType()).alias(\"PHOTOSOURCETYPE\")\n",
        "    )\n",
        ")\n",
        "\n",
        "staging_df.write.format(\"delta\").mode(\"overwrite\").saveAsTable(\"workspace.prxbi_dw.c_mercury_badge_ts_stg\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "staging_count = spark.table(\"workspace.prxbi_dw.c_mercury_badge_ts_stg\").count()\n",
        "print(f\"Staging table (c_mercury_badge_ts_stg) count: {staging_count}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Flow Table: i_wc_badge_details_d_flow"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {80}\n",
        "# Drop the flow table if it exists\n",
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow PURGE\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {90}, {100}\n",
        "# Build and write the flow DataFrame\n",
        "\n",
        "c_staging_df = spark.table(\"workspace.prxbi_dw.c_mercury_badge_ts_stg\")\n",
        "\n",
        "# Subquery for WC_BADGE_PRODUCT_D_2 deduplication\n",
        "wc_badge_product_dedup_df = (\n",
        "    spark.table(\"workspace.prxbi_dw.wc_badge_product_d\")\n",
        "    .withColumn(\n",
        "        \"COL\",\n",
        "        Window.partitionBy(\"SKU\").orderBy(F.col(\"ID\").desc()).rank()\n",
        "    )\n",
        "    .filter(F.col(\"COL\") == 1)\n",
        "    .select(\n",
        "        F.col(\"ID\").alias(\"PRODUCT_ID\"),\n",
        "        F.col(\"SKU\").alias(\"SKU\"),\n",
        "        F.col(\"NAME\").alias(\"NAME\"),\n",
        "        F.col(\"SKU\").alias(\"SKU_1\"),\n",
        "        F.col(\"NAME\").alias(\"NAME_1\")\n",
        "    )\n",
        ")\n",
        "\n",
        "# Join C$_ staging with WC_BADGE_PRODUCT_D_2\n",
        "joined_df = (\n",
        "    c_staging_df.alias(\"JOIN1_A\")\n",
        "    .join(\n",
        "        wc_badge_product_dedup_df.alias(\"WC_BADGE_PRODUCT_D_2\"),\n",
        "        F.col(\"JOIN1_A.PRODUCTCODE\") == F.col(\"WC_BADGE_PRODUCT_D_2.SKU_1\"),\n",
        "        \"left_outer\"\n",
        "    )\n",
        ")\n",
        "\n",
        "flow_select_df = (\n",
        "    joined_df.select(\n",
        "        F.col(\"JOIN1_A.ID\").alias(\"BADGE_ID\"),\n",
        "        F.col(\"JOIN1_A.BADGELOCATION\").alias(\"BADGE_LOCATION\"),\n",
        "        F.col(\"JOIN1_A.BADGETOKEN\").alias(\"BADGE_TOKEN\"),\n",
        "        F.col(\"JOIN1_A.BADGEVERSION\").alias(\"BADGE_VERSION\"),\n",
        "        F.col(\"JOIN1_A.CONTACTEMAIL\").alias(\"CONTACT_EMAIL\"),\n",
        "        F.col(\"JOIN1_A.CONTACTFIRSTNAME\").alias(\"CONTACT_FIRST_NAME\"),\n",
        "        F.col(\"JOIN1_A.CONTACTLASTNAME\").alias(\"CONTACT_LAST_NAME\"),\n",
        "        F.col(\"JOIN1_A.CONTACTJOBTITLE\").alias(\"CONTACT_JOB_TITLE\"),\n",
        "        F.col(\"JOIN1_A.CONTACTPERSONRXMASTERID\").alias(\"CONTACT_PERSON_ID\"),\n",
        "        F.col(\"JOIN1_A.CREATEDBYREGISTRATIONTYPE\").alias(\"CREATION_REG_TYPE\"),\n",
        "        F.col(\"JOIN1_A.CREATEDBYTYPE\").alias(\"CREATION_TYPE\"),\n",
        "        F.col(\"JOIN1_A.CULTURE\").alias(\"CULTURE\"),\n",
        "        F.col(\"JOIN1_A.CUSTOMERTYPE\").alias(\"CUSTOMER_TYPE\"),\n",
        "        F.col(\"JOIN1_A.EVENTEDITIONGBSCODE\").alias(\"EVENT_EDITION_CODE\"),\n",
        "        F.col(\"JOIN1_A.ISBADGEUPDATE\").alias(\"BADGE_UPDATE_FLG\"),\n",
        "        F.col(\"JOIN1_A.MARKETINGPREFERENCESPROMPTREQU\").alias(\"MARKETING_PREF_PROMPT\"),\n",
        "        F.col(\"JOIN1_A.ORGANISATIONDISPLAYNAME\").alias(\"ORG_NAME\"),\n",
        "        F.col(\"JOIN1_A.ORGANISATIONCITY\").alias(\"ORG_CITY\"),\n",
        "        F.col(\"JOIN1_A.ORGANISATIONCOUNTRYCODE\").alias(\"ORG_COUNTRY\"),\n",
        "        F.col(\"JOIN1_A.ORGANISATIONRXMASTERID\").alias(\"ORG_ID\"),\n",
        "        F.col(\"JOIN1_A.ORGANISATIONSTATE\").alias(\"ORG_STATE\"),\n",
        "        F.col(\"JOIN1_A.PARTICIPATINGORGANISATIONID\").alias(\"PARTICIPATING_ORG_ID\"),\n",
        "        F.col(\"JOIN1_A.PRODUCTCODE\").alias(\"PRODUCT_CODE\"),\n",
        "        F.col(\"JOIN1_A.QRCODECONTENT\").alias(\"QR_CODE\"),\n",
        "        F.col(\"JOIN1_A.REGISTRATIONID\").alias(\"REGISTRATION_ID\"),\n",
        "        F.col(\"JOIN1_A.STATUS\").alias(\"STATUS\"),\n",
        "        F.col(\"JOIN1_A.SUPPORTSTAFFCOMPANYNAME\").alias(\"STAFF_COMPANY_NAME\"),\n",
        "        F.col(\"JOIN1_A.SUPPORTSTAFFCOMPANYADDRESS\").alias(\"STAFF_COMPANY_ADDR\"),\n",
        "        F.col(\"JOIN1_A.SUPPORTSTAFFMOBILEPHONE\").alias(\"STAFF_PHONE_NUM\"),\n",
        "        F.col(\"JOIN1_A.SUPPORTSTAFFREPORTSTO\").alias(\"STAFF_REPORTING\"),\n",
        "        F.col(\"JOIN1_A.SUPPORTSTAFFSTANDS\").alias(\"STAFF_STANDS\"),\
        "        F.col(\"JOIN1_A.SUPPORTSTAFFUSERACCESS\").alias(\"STAFF_USER_ACCESS\"),\n",
        "        F.col(\"JOIN1_A.VERSIONNUMBER\").alias(\"VERSION_NUM\"),\n",
        "        F.col(\"JOIN1_A.ID\").alias(\"INTEGRATION_ID\"),\n",
        "        F.lit(datasource_num_id).alias(\"DATASOURCE_NUM_ID\"),\n",
        "        F.col(\"JOIN1_A.MOBILEPHONE\").alias(\"MOBILEPHONE\"),\n",
        "        F.col(\"JOIN1_A.FIRSTSCANNEDDATE\").alias(\"FIRSTSCANNEDDATE\"),\n",
        "        F.col(\"JOIN1_A.LASTPRINTEDDATE\").alias(\"LASTPRINTEDDATE\"),\n",
        "        F.when(F.col(\"JOIN1_A.FIRSTSCANNEDDATE\").isNotNull(), F.lit(\"Y\")).otherwise(F.lit(\"N\")).alias(\"FIRSTSCANNEDDATE_FLG\"),\n",
        "        F.when(F.col(\"JOIN1_A.LASTPRINTEDDATE\").isNotNull(), F.lit(\"Y\")).otherwise(F.lit(\"N\")).alias(\"LASTPRINTEDDATE_FLG\"),\n",
        "        F.col(\"JOIN1_A.ACCESSVALIDITYMODIFIEDDATE\").alias(\"ACCESSVALIDITYMODIFIEDDATE\"),\n",
        "        F.col(\"JOIN1_A.CREATEDDATE\").alias(\"CREATEDDATE\"),\n",
        "        F.col(\"JOIN1_A.COMPANYPRODUCTCODE\").alias(\"COMPANYPRODUCTCODE\"),\n",
        "        F.col(\"JOIN1_A.PAYMENTSTATUS\").alias(\"PAYMENTSTATUS\"),\n",
        "        F.col(\"JOIN1_A.PHOTOKEY\").alias(\"PHOTOKEY\"),\n",
        "        F.col(\"JOIN1_A.PHOTOSOURCE\").alias(\"PHOTOSOURCE\"),\n",
        "        F.col(\"JOIN1_A.PHOTOSOURCETYPE\").alias(\"PHOTOSOURCETYPE\"),\n",
        "        F.col(\"WC_BADGE_PRODUCT_D_2.NAME_1\").alias(\"PACKAGE_NAME\"),\n",
        "        F.lit(\"I\").alias(\"IND_UPDATE\") # Default to 'I' for insert\n",
        "    )\n",
        ")\n",
        "\n",
        "# Prepare target table for anti-join to detect existing records\n",
        "target_df_for_anti_join = (\n",
        "    spark.table(\"workspace.prxbi_dw.wc_badge_details_d\")\n",
        "    .select(\n",
        "        F.col(\"BADGE_ID\"), F.col(\"BADGE_LOCATION\"), F.col(\"BADGE_TOKEN\"), F.col(\"BADGE_VERSION\"),\n",
        "        F.col(\"CONTACT_EMAIL\"), F.col(\"CONTACT_FIRST_NAME\"), F.col(\"CONTACT_LAST_NAME\"), F.col(\"CONTACT_JOB_TITLE\"),\n",
        "        F.col(\"CONTACT_PERSON_ID\"), F.col(\"CREATION_REG_TYPE\"), F.col(\"CREATION_TYPE\"), F.col(\"CULTURE\"),\n",
        "        F.col(\"CUSTOMER_TYPE\"), F.col(\"EVENT_EDITION_CODE\"), F.col(\"BADGE_UPDATE_FLG\"), F.col(\"MARKETING_PREF_PROMPT\"),\n",
        "        F.col(\"ORG_NAME\"), F.col(\"ORG_CITY\"), F.col(\"ORG_COUNTRY\"), F.col(\"ORG_ID\"),\n",
        "        F.col(\"ORG_STATE\"), F.col(\"PARTICIPATING_ORG_ID\"), F.col(\"PRODUCT_CODE\"), F.col(\"QR_CODE\"),\n",
        "        F.col(\"REGISTRATION_ID\"), F.col(\"STATUS\"), F.col(\"STAFF_COMPANY_NAME\"), F.col(\"STAFF_COMPANY_ADDR\"),\n",
        "        F.col(\"STAFF_PHONE_NUM\"), F.col(\"STAFF_REPORTING\"), F.col(\"STAFF_STANDS\"), F.col(\"STAFF_USER_ACCESS\"),\n",
        "        F.col(\"VERSION_NUM\"), F.col(\"INTEGRATION_ID\"), F.col(\"DATASOURCE_NUM_ID\"), F.col(\"MOBILEPHONE\"),\n",
        "        F.col(\"FIRSTSCANNEDDATE\"), F.col(\"LASTPRINTEDDATE\"), F.col(\"FIRSTSCANNEDDATE_FLG\"), F.col(\"LASTPRINTEDDATE_FLG\"),\n",
        "        F.col(\"ACCESSVALIDITYMODIFIEDDATE\"), F.col(\"CREATEDDATE\"), F.col(\"COMPANYPRODUCTCODE\"), F.col(\"PAYMENTSTATUS\"),\n",
        "        F.col(\"PHOTOKEY\"), F.col(\"PHOTOSOURCE\"), F.col(\"PHOTOSOURCETYPE\"), F.col(\"PACKAGE_NAME\")\n",
        "    )\n",
        ")\n",
        "\n",
        "# Perform anti-join to filter out records that already exist in the target table based on all attributes\n",
        "# The complex join condition for NULL-safe equality implies we should join on the business keys and then filter\n",
        "# For the purpose of `NOT EXISTS` based on all columns, a `left_anti` join is appropriate if keys are matched first.\n",
        "# However, the ODI source has `NOT EXISTS` on `INTEGRATION_ID` and `DATASOURCE_NUM_ID` first, then the remaining columns.\n",
        "\n",
        "# Let's generate a full list of columns for NULL-safe equality comparison\n",
        "all_cols_for_comparison = [\n",
        "    \"BADGE_ID\", \"BADGE_LOCATION\", \"BADGE_TOKEN\", \"BADGE_VERSION\", \"CONTACT_EMAIL\",\n",
        "    \"CONTACT_FIRST_NAME\", \"CONTACT_LAST_NAME\", \"CONTACT_JOB_TITLE\", \"CONTACT_PERSON_ID\",\n",
        "    \"CREATION_REG_TYPE\", \"CREATION_TYPE\", \"CULTURE\", \"CUSTOMER_TYPE\", \"EVENT_EDITION_CODE\",\n",
        "    \"BADGE_UPDATE_FLG\", \"MARKETING_PREF_PROMPT\", \"ORG_NAME\", \"ORG_CITY\", \"ORG_COUNTRY\",\n",
        "    \"ORG_ID\", \"ORG_STATE\", \"PARTICIPATING_ORG_ID\", \"PRODUCT_CODE\", \"QR_CODE\",\n",
        "    \"REGISTRATION_ID\", \"STATUS\", \"STAFF_COMPANY_NAME\", \"STAFF_COMPANY_ADDR\",\n",
        "    \"STAFF_PHONE_NUM\", \"STAFF_REPORTING\", \"STAFF_STANDS\", \"STAFF_USER_ACCESS\",\n",
        "    \"VERSION_NUM\", \"MOBILEPHONE\", \"FIRSTSCANNEDDATE\", \"LASTPRINTEDDATE\",\n",
        "    \"FIRSTSCANNEDDATE_FLG\", \"LASTPRINTEDDATE_FLG\", \"ACCESSVALIDITYMODIFIEDDATE\",\n",
        "    \"CREATEDDATE\", \"COMPANYPRODUCTCODE\", \"PAYMENTSTATUS\", \"PHOTOKEY\",\n",
        "    \"PHOTOSOURCE\", \"PHOTOSOURCETYPE\", \"PACKAGE_NAME\"\n",
        "]\n",
        "\n",
        "join_condition = (\n",
        "    (F.col(\"S.INTEGRATION_ID\") == F.col(\"T.INTEGRATION_ID\")) &\n",
        "    (F.col(\"S.DATASOURCE_NUM_ID\") == F.col(\"T.DATASOURCE_NUM_ID\"))\n",
        ")\n",
        "\n",
        "for col_name in all_cols_for_comparison:\n",
        "    join_condition = join_condition & (\n",
        "        (F.col(f\"S.{col_name}\") == F.col(f\"T.{col_name}\")) |\n",
        "        (F.col(f\"S.{col_name}\").isNull() & F.col(f\"T.{col_name}\").isNull())\n",
        "    )\n",
        "\n",
        "flow_df_insert = (\n",
        "    flow_select_df.alias(\"S\")\n",
        "    .join(target_df_for_anti_join.alias(\"T\"), join_condition, \"left_anti\")\n",
        "    .select(\n",
        "        F.col(\"S.BADGE_ID\"), F.col(\"S.BADGE_LOCATION\"), F.col(\"S.BADGE_TOKEN\"), F.col(\"S.BADGE_VERSION\"),\n",
        "        F.col(\"S.CONTACT_EMAIL\"), F.col(\"S.CONTACT_FIRST_NAME\"), F.col(\"S.CONTACT_LAST_NAME\"), F.col(\"S.CONTACT_JOB_TITLE\"),\n",
        "        F.col(\"S.CONTACT_PERSON_ID\"), F.col(\"S.CREATION_REG_TYPE\"), F.col(\"S.CREATION_TYPE\"), F.col(\"S.CULTURE\"),\n",
        "        F.col(\"S.CUSTOMER_TYPE\"), F.col(\"S.EVENT_EDITION_CODE\"), F.col(\"S.BADGE_UPDATE_FLG\"), F.col(\"S.MARKETING_PREF_PROMPT\"),\n",
        "        F.col(\"S.ORG_NAME\"), F.col(\"S.ORG_CITY\"), F.col(\"S.ORG_COUNTRY\"), F.col(\"S.ORG_ID\"),\n",
        "        F.col(\"S.ORG_STATE\"), F.col(\"S.PARTICIPATING_ORG_ID\"), F.col(\"S.PRODUCT_CODE\"), F.col(\"S.QR_CODE\"),\n",
        "        F.col(\"S.REGISTRATION_ID\"), F.col(\"S.STATUS\"), F.col(\"S.STAFF_COMPANY_NAME\"), F.col(\"S.STAFF_COMPANY_ADDR\"),\n",
        "        F.col(\"S.STAFF_PHONE_NUM\"), F.col(\"S.STAFF_REPORTING\"), F.col(\"S.STAFF_STANDS\"), F.col(\"S.STAFF_USER_ACCESS\"),\n",
        "        F.col(\"S.VERSION_NUM\"), F.col(\"S.INTEGRATION_ID\"), F.col(\"S.DATASOURCE_NUM_ID\"), F.col(\"S.MOBILEPHONE\"),\n",
        "        F.col(\"S.FIRSTSCANNEDDATE\"), F.col(\"S.LASTPRINTEDDATE\"), F.col(\"S.FIRSTSCANNEDDATE_FLG\"), F.col(\"S.LASTPRINTEDDATE_FLG\"),\n",
        "        F.col(\"S.ACCESSVALIDITYMODIFIEDDATE\"), F.col(\"S.CREATEDDATE\"), F.col(\"S.COMPANYPRODUCTCODE\"), F.col(\"S.PAYMENTSTATUS\"),\n",
        "        F.col(\"S.PHOTOKEY\"), F.col(\"S.PHOTOSOURCE\"), F.col(\"S.PHOTOSOURCETYPE\"), F.col(\"S.PACKAGE_NAME\"),\n",
        "        F.col(\"S.IND_UPDATE\")\n",
        "    )\n",
        ")\n",
        "\n",
        "flow_df_insert.write.format(\"delta\").mode(\"overwrite\").saveAsTable(\"workspace.prxbi_dw.i_wc_badge_details_d_flow\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "flow_count = spark.table(\"workspace.prxbi_dw.i_wc_badge_details_d_flow\").count()\n",
        "print(f\"Flow table (i_wc_badge_details_d_flow) count: {flow_count}\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {110}, {120}\n",
        "# Optimize the flow table\n",
        "spark.sql(\"SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false\")\n",
        "spark.sql(\"OPTIMIZE workspace.prxbi_dw.i_wc_badge_details_d_flow ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID)\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Error / Audit Tables"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {15}\n",
        "# Placeholder for E$ table creation if needed. The original script does not show E$ table creation or usage.\n",
        "# Assuming no E$ table for this specific script based on the provided tasks."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {16}\n",
        "# No explicit DELETE from E$ table by ODI_SESS_NO in the provided script."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {17}\n",
        "# No explicit CREATE TABLE for snp_check_tab in the provided script."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {18}\n",
        "# No explicit DELETE from snp_check_tab in the provided script."
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## PK Violation Detection"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {20}\n",
        "# No explicit PK violation detection and insertion into E$ in the provided script."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {21}\n",
        "# No explicit deduplication of flow table using ROW_NUMBER overwrite, as the initial load for flow_df_insert already handles 'NOT EXISTS'."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {22}\n",
        "# No explicit summary insertion into snp_check_tab in the provided script."
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Mark Records for Update"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {130}\n",
        "# Mark records in the flow table (i_wc_badge_details_d_flow) for update ('U')\n",
        "DeltaTable.forName(spark, \"workspace.prxbi_dw.i_wc_badge_details_d_flow\").alias(\"t\").merge(\n",
        "    spark.table(\"workspace.prxbi_dw.wc_badge_details_d\")\n",
        "    .select(\"INTEGRATION_ID\", \"DATASOURCE_NUM_ID\")\n",
        "    .alias(\"s\"),\n",
        "    \"t.INTEGRATION_ID = s.INTEGRATION_ID AND t.DATASOURCE_NUM_ID = s.DATASOURCE_NUM_ID\"\n",
        ").whenMatchedUpdate(set={\n",
        "    \"t.IND_UPDATE\": F.lit(\"U\")\n",
        "}).execute()"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Merge into Target: workspace.prxbi_dw.wc_badge_details_d"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {150} (UPDATE) and {160} (INSERT)\n",
        "# Perform a single merge operation into the target table\n",
        "\n",
        "target_merge_df = DeltaTable.forName(spark, \"workspace.prxbi_dw.wc_badge_details_d\")\n",
        "\n",
        "target_merge_df.alias(\"t\").merge(\n",
        "    spark.table(\"workspace.prxbi_dw.i_wc_badge_details_d_flow\").alias(\"s\"),\n",
        "    \"t.INTEGRATION_ID = s.INTEGRATION_ID AND t.DATASOURCE_NUM_ID = s.DATASOURCE_NUM_ID\"\n",
        ").whenMatchedUpdate(\n",
        "    condition=\"s.IND_UPDATE = 'U'\",\n",
        "    set={\n",
        "        \"t.BADGE_ID\": F.col(\"s.BADGE_ID\"),\n",
        "        \"t.BADGE_LOCATION\": F.col(\"s.BADGE_LOCATION\"),\n",
        "        \"t.BADGE_TOKEN\": F.col(\"s.BADGE_TOKEN\"),\n",
        "        \"t.BADGE_VERSION\": F.col(\"s.BADGE_VERSION\"),\n",
        "        \"t.CONTACT_EMAIL\": F.col(\"s.CONTACT_EMAIL\"),\n",
        "        \"t.CONTACT_FIRST_NAME\": F.col(\"s.CONTACT_FIRST_NAME\"),\n",
        "        \"t.CONTACT_LAST_NAME\": F.col(\"s.CONTACT_LAST_NAME\"),\n",
        "        \"t.CONTACT_JOB_TITLE\": F.col(\"s.CONTACT_JOB_TITLE\"),\n",
        "        \"t.CONTACT_PERSON_ID\": F.col(\"s.CONTACT_PERSON_ID\"),\n",
        "        \"t.CREATION_REG_TYPE\": F.col(\"s.CREATION_REG_TYPE\"),\n",
        "        \"t.CREATION_TYPE\": F.col(\"s.CREATION_TYPE\"),\n",
        "        \"t.CULTURE\": F.col(\"s.CULTURE\"),\n",
        "        \"t.CUSTOMER_TYPE\": F.col(\"s.CUSTOMER_TYPE\"),\n",
        "        \"t.EVENT_EDITION_CODE\": F.col(\"s.EVENT_EDITION_CODE\"),\n",
        "        \"t.BADGE_UPDATE_FLG\": F.col(\"s.BADGE_UPDATE_FLG\"),\n",
        "        \"t.MARKETING_PREF_PROMPT\": F.col(\"s.MARKETING_PREF_PROMPT\"),\n",
        "        \"t.ORG_NAME\": F.col(\"s.ORG_NAME\"),\n",
        "        \"t.ORG_CITY\": F.col(\"s.ORG_CITY\"),\n",
        "        \"t.ORG_COUNTRY\": F.col(\"s.ORG_COUNTRY\"),\n",
        "        \"t.ORG_ID\": F.col(\"s.ORG_ID\"),\n",
        "        \"t.ORG_STATE\": F.col(\"s.ORG_STATE\"),\n",
        "        \"t.PARTICIPATING_ORG_ID\": F.col(\"s.PARTICIPATING_ORG_ID\"),\n",
        "        \"t.PRODUCT_CODE\": F.col(\"s.PRODUCT_CODE\"),\n",
        "        \"t.QR_CODE\": F.col(\"s.QR_CODE\"),\n",
        "        \"t.REGISTRATION_ID\": F.col(\"s.REGISTRATION_ID\"),\n",
        "        \"t.STATUS\": F.col(\"s.STATUS\"),\n",
        "        \"t.STAFF_COMPANY_NAME\": F.col(\"s.STAFF_COMPANY_NAME\"),\n",
        "        \"t.STAFF_COMPANY_ADDR\": F.col(\"s.STAFF_COMPANY_ADDR\"),\n",
        "        \"t.STAFF_PHONE_NUM\": F.col(\"s.STAFF_PHONE_NUM\"),\n",
        "        \"t.STAFF_REPORTING\": F.col(\"s.STAFF_REPORTING\"),\n",
        "        \"t.STAFF_STANDS\": F.col(\"s.STAFF_STANDS\"),\n",
        "        \"t.STAFF_USER_ACCESS\": F.col(\"s.STAFF_USER_ACCESS\"),\n",
        "        \"t.VERSION_NUM\": F.col(\"s.VERSION_NUM\"),\n",
        "        \"t.MOBILEPHONE\": F.col(\"s.MOBILEPHONE\"),\n",
        "        \"t.FIRSTSCANNEDDATE\": F.col(\"s.FIRSTSCANNEDDATE\"),\n",
        "        \"t.LASTPRINTEDDATE\": F.col(\"s.LASTPRINTEDDATE\"),\
        "        \"t.FIRSTSCANNEDDATE_FLG\": F.col(\"s.FIRSTSCANNEDDATE_FLG\"),\n",
        "        \"t.LASTPRINTEDDATE_FLG\": F.col(\"s.LASTPRINTEDDATE_FLG\"),\n",
        "        \"t.ACCESSVALIDITYMODIFIEDDATE\": F.col(\"s.ACCESSVALIDITYMODIFIEDDATE\"),\n",
        "        \"t.CREATEDDATE\": F.col(\"s.CREATEDDATE\"),\n",
        "        \"t.COMPANYPRODUCTCODE\": F.col(\"s.COMPANYPRODUCTCODE\"),\n",
        "        \"t.PAYMENTSTATUS\": F.col(\"s.PAYMENTSTATUS\"),\n",
        "        \"t.PHOTOKEY\": F.col(\"s.PHOTOKEY\"),\n",
        "        \"t.PHOTOSOURCE\": F.col(\"s.PHOTOSOURCE\"),\n",
        "        \"t.PHOTOSOURCETYPE\": F.col(\"s.PHOTOSOURCETYPE\"),\n",
        "        \"t.PACKAGE_NAME\": F.col(\"s.PACKAGE_NAME\"),\n",
        "        \"t.W_UPDATE_DT\": F.current_timestamp()\n",
        "    }\n",
        ").whenNotMatchedInsert(\n",
        "    condition=\"s.IND_UPDATE = 'I'\",\n",
        "    values={\n",
        "        \"BADGE_ID\": F.col(\"s.BADGE_ID\"),\n",
        "        \"BADGE_LOCATION\": F.col(\"s.BADGE_LOCATION\"),\n",
        "        \"BADGE_TOKEN\": F.col(\"s.BADGE_TOKEN\"),\n",
        "        \"BADGE_VERSION\": F.col(\"s.BADGE_VERSION\"),\n",
        "        \"CONTACT_EMAIL\": F.col(\"s.CONTACT_EMAIL\"),\n",
        "        \"CONTACT_FIRST_NAME\": F.col(\"s.CONTACT_FIRST_NAME\"),\n",
        "        \"CONTACT_LAST_NAME\": F.col(\"s.CONTACT_LAST_NAME\"),\n",
        "        \"CONTACT_JOB_TITLE\": F.col(\"s.CONTACT_JOB_TITLE\"),\n",
        "        \"CONTACT_PERSON_ID\": F.col(\"s.CONTACT_PERSON_ID\"),\n",
        "        \"CREATION_REG_TYPE\": F.col(\"s.CREATION_REG_TYPE\"),\n",
        "        \"CREATION_TYPE\": F.col(\"s.CREATION_TYPE\"),\n",
        "        \"CULTURE\": F.col(\"s.CULTURE\"),\n",
        "        \"CUSTOMER_TYPE\": F.col(\"s.CUSTOMER_TYPE\"),\n",
        "        \"EVENT_EDITION_CODE\": F.col(\"s.EVENT_EDITION_CODE\"),\n",
        "        \"BADGE_UPDATE_FLG\": F.col(\"s.BADGE_UPDATE_FLG\"),\n",
        "        \"MARKETING_PREF_PROMPT\": F.col(\"s.MARKETING_PREF_PROMPT\"),\n",
        "        \"ORG_NAME\": F.col(\"s.ORG_NAME\"),\n",
        "        \"ORG_CITY\": F.col(\"s.ORG_CITY\"),\n",
        "        \"ORG_COUNTRY\": F.col(\"s.ORG_COUNTRY\"),\n",
        "        \"ORG_ID\": F.col(\"s.ORG_ID\"),\n",
        "        \"ORG_STATE\": F.col(\"s.ORG_STATE\"),\n",
        "        \"PARTICIPATING_ORG_ID\": F.col(\"s.PARTICIPATING_ORG_ID\"),\n",
        "        \"PRODUCT_CODE\": F.col(\"s.PRODUCT_CODE\"),\n",
        "        \"QR_CODE\": F.col(\"s.QR_CODE\"),\
        "        \"REGISTRATION_ID\": F.col(\"s.REGISTRATION_ID\"),\n",
        "        \"STATUS\": F.col(\"s.STATUS\"),\n",
        "        \"STAFF_COMPANY_NAME\": F.col(\"s.STAFF_COMPANY_NAME\"),\n",
        "        \"STAFF_COMPANY_ADDR\": F.col(\"s.STAFF_COMPANY_ADDR\"),\n",
        "        \"STAFF_PHONE_NUM\": F.col(\"s.STAFF_PHONE_NUM\"),\n",
        "        \"STAFF_REPORTING\": F.col(\"s.STAFF_REPORTING\"),\n",
        "        \"STAFF_STANDS\": F.col(\"s.STAFF_STANDS\"),\n",
        "        \"STAFF_USER_ACCESS\": F.col(\"s.STAFF_USER_ACCESS\"),\n",
        "        \"VERSION_NUM\": F.col(\"s.VERSION_NUM\"),\n",
        "        \"INTEGRATION_ID\": F.col(\"s.INTEGRATION_ID\"),\n",
        "        \"DATASOURCE_NUM_ID\": F.col(\"s.DATASOURCE_NUM_ID\"),\n",
        "        \"MOBILEPHONE\": F.col(\"s.MOBILEPHONE\"),\n",
        "        \"FIRSTSCANNEDDATE\": F.col(\"s.FIRSTSCANNEDDATE\"),\n",
        "        \"LASTPRINTEDDATE\": F.col(\"s.LASTPRINTEDDATE\"),\n",
        "        \"FIRSTSCANNEDDATE_FLG\": F.col(\"s.FIRSTSCANNEDDATE_FLG\"),\n",
        "        \"LASTPRINTEDDATE_FLG\": F.col(\"s.LASTPRINTEDDATE_FLG\"),\n",
        "        \"ACCESSVALIDITYMODIFIEDDATE\": F.col(\"s.ACCESSVALIDITYMODIFIEDDATE\"),\n",
        "        \"CREATEDDATE\": F.col(\"s.CREATEDDATE\"),\n",
        "        \"COMPANYPRODUCTCODE\": F.col(\"s.COMPANYPRODUCTCODE\"),\n",
        "        \"PAYMENTSTATUS\": F.col(\"s.PAYMENTSTATUS\"),\n",
        "        \"PHOTOKEY\": F.col(\"s.PHOTOKEY\"),\n",
        "        \"PHOTOSOURCE\": F.col(\"s.PHOTOSOURCE\"),\n",
        "        \"PHOTOSOURCETYPE\": F.col(\"s.PHOTOSOURCETYPE\"),\n",
        "        \"PACKAGE_NAME\": F.col(\"s.PACKAGE_NAME\"),\n",
        "        \"W_INSERT_DT\": F.current_timestamp(),\n",
        "        \"W_UPDATE_DT\": F.current_timestamp()\n",
        "    }\n",
        ").execute()"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Optimize Target"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {170} (COMMIT - not applicable), {190} (No DBMS_STATS shown after merge)\n",
        "# Optimize the target table\n",
        "spark.sql(\"SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false\")\n",
        "spark.sql(\"OPTIMIZE workspace.prxbi_dw.wc_badge_details_d ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID)\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Cleanup"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "# SCEN_TASK_NO {180}, {210}\n",
        "# Drop staging and flow tables\n",
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow PURGE\")\n",
        "spark.sql(\"DROP TABLE IF EXISTS workspace.prxbi_dw.c_mercury_badge_ts_stg PURGE\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Validation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "final_target_count = spark.table(\"workspace.prxbi_dw.wc_badge_details_d\").count()\n",
        "print(f\"Final target table (wc_badge_details_d) count: {final_target_count}\")\n",
        "\n",
        "print(\"Sample of target data:\")\n",
        "display(spark.table(\"workspace.prxbi_dw.wc_badge_details_d\").limit(10))"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "spark.stop()"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "ipython",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.10.12"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}